# 08 - Đối chiếu composite score với Pareto front độc lập (Nhóm 3.2)

**Bối cảnh** (`docs/PROJECT_PLAN.md` Nhóm 3.2, "làm song song 4.1/4.2"): `best_of_n_eval.py` chọn ứng viên tốt nhất trong N mẫu sinh ra cho 1 target bằng `composite_score = 0.6*accuracy + 0.3*manuf + 0.1*aesthetic` (trọng số cố định). Câu hỏi cần trả lời: công thức gộp có tuyến tính này có **tôn trọng đúng cấu trúc đánh đổi đa mục tiêu thật** giữa 3 điểm số, hay chỉ là một cách chọn tùy tiện?

**Phương pháp** (độc lập với composite score, tái dùng `analysis/pareto/frontier.py::is_pareto_efficient()` có sẵn từ Phase 1, không viết thuật toán Pareto mới):
1. Với 24 target (seed=123, cùng quy ước mọi benchmark khác trong dự án), sinh N=30 ứng viên/target bằng `cvae_v2_finetuned.pt` (checkpoint production khuyến nghị, `docs/PIPELINE.md`).
2. Với mỗi target, xếp hạng 30 ứng viên theo **tầng Pareto** (NSGA-II style: tầng 1 = không bị ứng viên nào khác trội hơn cả 3 mặt (accuracy, manuf, aesthetic); tầng 2 = phần còn lại sau khi bỏ tầng 1; v.v.) - cách xếp hạng này KHÔNG dùng trọng số 0.6/0.3/0.1, thuần túy dựa trên quan hệ trội (dominance).
3. Tính hệ số tương quan Spearman giữa `composite_score` và `-tầng_Pareto` (tầng thấp = tốt) cho từng target, rồi lấy trung bình/trung vị trên 24 target.

**Tiêu chí dừng đặt TRƯỚC khi chạy** (tránh diễn giải có lợi sau khi có kết quả, theo đúng yêu cầu Nhóm 3.2): **Spearman trung bình ≥ 0.7** thì coi composite score tôn trọng tốt cấu trúc Pareto; thấp hơn là dấu hiệu cần xem lại trọng số.

**Phát hiện phụ trong lúc làm** (xem `EXPERIMENT_LOG.md`, `docs/LIMITATIONS.md`): `is_pareto_efficient()` có bug thật - luôn đánh dấu SAI phần tử CUỐI CÙNG của mảng đầu vào là không-Pareto-efficient (kể cả khi đó là điểm tốt nhất tuyệt đối), gây vòng lặp vô hạn khi dùng lặp lại để xếp tầng. Đã sửa + thêm test trước khi chạy notebook này (`tests/test_pareto_frontier.py`, 8 test mới). Module `analysis/pareto/` chưa từng chạy thật cho kết quả nào đã công bố trước đây nên không có claim khoa học cũ nào bị ảnh hưởng.

In [1]:
import os
import sys
import json
import time

import numpy as np
import torch
from scipy.stats import spearmanr

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "pipeline", "phase5_cvae"))
sys.path.insert(0, REPO_ROOT)

from best_of_n_eval import best_of_n              # noqa: E402
from dataset import CVAEDataset                    # noqa: E402
from analysis.pareto.frontier import is_pareto_efficient  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = os.path.join(REPO_ROOT, "outputs", "phase5", "cvae_v2_finetuned.pt")
print("device:", device, "| checkpoint:", CKPT)

In [2]:
def non_domination_ranks(costs: np.ndarray, maximize: bool = True) -> np.ndarray:
    """NSGA-II style: tang 1 = Pareto front, tang 2 = phan con lai sau khi
    bo tang 1, v.v. Tra ve mang rank (1-indexed) cung do dai voi costs.
    Dua tren is_pareto_efficient() da sua bug bien (xem markdown tren)."""
    n = costs.shape[0]
    ranks = np.zeros(n, dtype=int)
    remaining = np.arange(n)
    rank = 1
    while len(remaining) > 0:
        mask = is_pareto_efficient(costs[remaining], maximize=maximize)
        ranks[remaining[mask]] = rank
        remaining = remaining[~mask]
        rank += 1
    return ranks


# 24 target that tu test.npz (seed=123, khop moi benchmark khac trong du an)
test_ds = CVAEDataset(os.path.join(REPO_ROOT, "outputs", "phase3", "test.npz"))
rng = np.random.default_rng(123)
idxs = rng.choice(len(test_ds), size=24, replace=False)
conditions = [test_ds[i][1].numpy() for i in idxs]

per_condition = []
for i, cond in enumerate(conditions):
    t0 = time.time()
    r = best_of_n(CKPT, n_conditions=1, n_samples=30, device=device, seed=123,
                  custom_condition=cond, return_all_scores=True)
    print(f"[{i+1}/24] v12_target={cond[0]:.3f} elapsed={time.time()-t0:.2f}s")
    if r["per_condition"]:
        per_condition.append(r["per_condition"][0])

In [3]:
spearman_per_condition = []
winner_on_front1 = []
for pc in per_condition:
    sc = pc["all_scores"]
    costs = np.stack([sc["accuracy"], sc["manuf"], sc["aesthetic"]], axis=1)
    if costs.shape[0] < 3:
        continue
    ranks = non_domination_ranks(costs, maximize=True)
    composite = np.array(sc["composite"])
    rho, _ = spearmanr(composite, -ranks)
    spearman_per_condition.append(rho)
    best_idx = int(np.argmax(composite))
    winner_on_front1.append(bool(ranks[best_idx] == 1))

spearman_per_condition = np.array(spearman_per_condition, dtype=float)
mean_spearman = float(np.nanmean(spearman_per_condition))
median_spearman = float(np.nanmedian(spearman_per_condition))
frac_winner_front1 = float(np.mean(winner_on_front1))

print("n conditions used:", len(spearman_per_condition))
print("spearman per condition:", np.round(spearman_per_condition, 3).tolist())
print("mean spearman:", mean_spearman)
print("median spearman:", median_spearman)
print("min/max spearman:", float(spearman_per_condition.min()), float(spearman_per_condition.max()))
print("frac winner on pareto front1:", frac_winner_front1)

## Kết quả thật (chạy 2026-08-20, `cvae_v2_finetuned.pt`, n_conditions=24, n_samples=30, seed=123)

```
n conditions used: 24
mean spearman:   0.6926
median spearman: 0.7137
min/max spearman: 0.4316 / 0.8886
frac winner on pareto front1: 1.0
```

**Đối chiếu với tiêu chí đã đặt trước (≥0.7):** **KHÔNG đạt theo trung bình** (0.693 < 0.7, sát ngưỡng), nhưng **đạt theo trung vị** (0.714 ≥ 0.7). Phân phối trải khá liên tục từ 0.43 đến 0.89 (24 target), không có nhóm outlier tách biệt rõ - một vài target có tương quan yếu hơn (0.43-0.56) kéo trung bình xuống dưới ngưỡng, phần lớn còn lại nằm trong khoảng 0.65-0.89.

**`frac_winner_on_pareto_front1 = 1.0`** (100% ứng viên thắng cuộc đều nằm ở tầng Pareto 1) - đây là **kết quả TẤT YẾU về mặt toán học, không phải bằng chứng thực nghiệm**: với `w_accuracy, w_manuf, w_aesthetic` đều > 0 (0.6/0.3/0.1), điểm argmax của một tổng có trọng số dương LUÔN nằm trên Pareto front (nếu 1 điểm khác trội hơn nó ở cả 3 trục, tổng có trọng số của điểm đó cũng phải cao hơn - mâu thuẫn với giả thiết argmax). Không nên trích dẫn con số này như một phát hiện, chỉ là xác nhận công thức composite score không vi phạm định lý cơ bản.

**Kết luận cẩn trọng (không phóng đại theo hướng nào):** composite score có tương quan DƯƠNG rõ ràng và ở mức trung bình-khá với cấu trúc Pareto thật (không phải ngẫu nhiên - toàn bộ 24/24 target đều dương, phần lớn >0.6), nhưng KHÔNG đạt ngưỡng "rất mạnh" (≥0.7) đã tự đặt ra theo tiêu chí trung bình - chỉ đạt theo trung vị. Đây là bằng chứng **ủng hộ một phần** cho việc composite score là một cách tổng hợp hợp lý (không phải tùy tiện), nhưng không đủ mạnh để tuyên bố "đã xác nhận" phương pháp luận xếp hạng đa mục tiêu mà không có dè dặt. Có thể do: (a) trọng số 0.6/0.3/0.1 chưa tối ưu, (b) accuracy_score chuẩn hóa min-max NGAY TRONG pool (rank-based, phụ thuộc phân phối |Δv12| của batch đó) tạo nhiễu khác với cấu trúc dominance tuyệt đối mà Pareto front đo. Chưa đủ cơ sở để đổi trọng số mặc định chỉ từ kết quả này - cần thêm dữ liệu hoặc thử nghiệm trọng số khác nếu muốn cải thiện.

Kết quả đầy đủ (JSON, per-condition): `outputs/phase5/reports/composite_score_pareto_validation.json`. Chi tiết đầy đủ + phát hiện bug `is_pareto_efficient()`: `EXPERIMENT_LOG.md`, mục 2026-08-20.